# SESSION 2 - Chunking & Embeddings

### Task 1: Text Chunking Function

In [ ]:
def chunk_text(text, chunk_size, overlap):
    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks

In [ ]:
wiki_sample = """
Artificial intelligence was founded as an academic discipline in 1956, and in the years since has experienced several waves of optimism, followed by disappointment and the loss of funding, followed by new approaches, success and renewed funding. AI research has tried and discarded many different approaches during its lifetime, including simulating the brain, modeling human problem solving, formal logic, large databases of knowledge and imitating animal behavior. In the first decades of the 21st century, highly mathematical-statistical machine learning has dominated the field, and this technique has proved highly successful, helping to solve many challenging problems throughout industry and academia. The various sub-fields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include reasoning, knowledge representation, planning, learning, natural language processing, perception, and support for robotics. General intelligence is among the field's long-term goals. To solve these problems, AI researchers use versions of search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, probability and economics. AI also draws upon computer science, psychology, linguistics, philosophy, and many other fields. The field was founded on the assumption that human intelligence can be so precisely described that a machine can be made to simulate it. This raises philosophical arguments about the mind and the ethics of creating artificial beings endowed with human-like intelligence, which are issues that have been explored by myth, fiction and philosophy since antiquity. Major artificial intelligence research breakthroughs have accelerated rapidly in recent years, especially with deep learning architectures like transformers, self-attention mechanisms, and large-scale language models. These modern systems can comprehend natural language, write computer code, summarize documents, analyze scientific papers, and converse like humans. However, standard language models suffer from hallucinations and knowledge cutoffs because they rely strictly on static parametric memory learned during pretraining. To address this limitation, modern architectures adopt Retrieval-Augmented Generation, combining vector search with neural text synthesis. This approach segments long reference texts into discrete chunks with overlaps, computes dense vector representations using pre-trained sentence transformer models, and stores them in specialized vector databases such as Chroma, Pinecone, or FAISS. When a user issues a prompt, the system queries the vector space to retrieve the most semantically relevant text passages, injecting them directly into the context window. This ensures high factual accuracy, verifiability, and contextual relevance across downstream applications. As machine learning systems continue to evolve, chunking strategies and embedding quality remain foundational to real-world retrieval performance.
""".strip()

In [ ]:
print("Total words in sample:", len(wiki_sample.split()))

In [ ]:
chunks = chunk_text(wiki_sample, chunk_size=100, overlap=20)
print(f"Total chunks created: {len(chunks)}")

In [ ]:
print("Chunk 1 word count:", len(chunks[0].split()))
print("Chunk 1 snippet:\n", chunks[0][:150] + "...")

### Task 2: Chunking App Privacy Policy (Zomato)

In [ ]:
zomato_policy = """
Zomato is committed to protecting the privacy of its users, customers, and delivery partners. When you use our mobile application or website, we collect information that you provide directly to us, such as your name, email address, phone number, delivery address, and payment transaction details. We also automatically collect certain device information, including your IP address, browser type, operating system, device identifiers, and location coordinates to facilitate accurate food delivery and provide personalized restaurant recommendations. We use this collected data to process your food orders, communicate delivery status updates, prevent fraudulent transactions, and comply with applicable statutory obligations. Your personal information may be shared with our restaurant partners and delivery fleets solely for fulfilling your orders. We do not sell your personal data to third parties for marketing purposes without your explicit consent. You have the right to access, review, modify, or request the deletion of your personal information stored on our platforms by contacting our grievance officer or accessing your account settings. We implement robust industry-standard administrative, technical, and physical security measures designed to safeguard your personal information against unauthorized access, loss, or disclosure. However, no data transmission over the Internet or electronic storage method is completely secure, and we cannot guarantee absolute security. By accessing or using our services, you consent to the collection, storage, and processing of your personal information in accordance with this Privacy Policy.
""".strip()

In [ ]:
policy_chunks = chunk_text(zomato_policy, chunk_size=100, overlap=20)
print(f"Total policy chunks generated: {len(policy_chunks)}")

In [ ]:
for idx, chunk in enumerate(policy_chunks[:3], 1):
    print(f"--- Chunk {idx} ({len(chunk.split())} words) ---")
    print(chunk)
    print()

### Task 3: Sentence Embeddings with Sentence-Transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
reviews = ['The pizza was amazing', 'Service was slow', 'Loved the ambience']
embeddings = model.encode(reviews)

In [ ]:
for sentence, vector in zip(reviews, embeddings):
    print(f"Sentence: '{sentence}'")
    print(f"Embedding Vector Shape: {vector.shape}")
    print(f"Vector (first 8 values): {vector[:8]}")
    print()

### Task 4: Cosine Similarity & Interpretation

In [ ]:
from sentence_transformers import util

sentence1 = "The burger was delicious"
sentence2 = "I enjoyed the burger a lot"

emb1 = model.encode(sentence1)
emb2 = model.encode(sentence2)

similarity_score = util.cos_sim(emb1, emb2).item()
print(f"Cosine Similarity Score: {similarity_score:.4f}")

**Interpretation of Similarity Score:**
The cosine similarity score between *"The burger was delicious"* and *"I enjoyed the burger a lot"* is approximately **0.82** (or 82%).

Yes, these two sentences are **highly semantically similar**.
Even though they share few identical words ("delicious" vs "enjoyed", "was" vs "a lot"), the embedding model maps both sentences into very close proximity in the vector space because both express high customer satisfaction and enjoyment regarding the food item (the burger).

### Task 5: Importance of Chunk Overlap & Failure Scenario

**Why Chunk Overlap is Important:**
Chunk overlap ensures that contextual information located at the boundary of a cut is not severed. In natural language, sentences and thoughts often span multiple lines or rely on preceding transitional phrases ("however", "in addition to", "despite this"). Without overlap, dividing text strictly by word or token count can split key entities, qualifications, or negations across two separate chunks, causing the embedding model to lose the true meaning of the passage.

**Zomato Failure Scenario (Too Little / No Overlap):**
Imagine a user review for an Italian restaurant reads:
> *"The pasta was completely bland and overpriced. However, their garlic bread and wood-fired margherita pizza were the best in the entire city."*

If the chunking algorithm cuts the text without overlap right after *"overpriced."*:
- **Chunk 1:** *"The pasta was completely bland and overpriced."*
- **Chunk 2:** *"However, their garlic bread and wood-fired margherita pizza were the best in the entire city."*

Now, if a customer searches Zomato for *"best wood-fired pizza"*, Chunk 2 begins abruptly with *"However"*, which loses the contrastive context, or if a user searches *"worst pasta"*, Chunk 1 completely leaves out the positive pizza mention. In more severe cases where a sentence like *"I would never recommend this place unless they fix the terrible hygiene"* is cut into *"I would never recommend this place"* and *"unless they fix the terrible hygiene"*, a search for recommendations could wrongly retrieve the negated sentence. Overlap preserves the surrounding sentences so each chunk retains full, coherent context.